Testing the final tuned model

In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import json
from sklearn.metrics import recall_score, precision_score, fbeta_score, confusion_matrix

print("--- 1. LOADING UNSEEN DATA (HOSPITAL B) ---")
df_B = pd.read_parquet('../data/processed/test_hospital_B.parquet')

# Drop columns that are all NaN (like EtCO2 in training)
df_B = df_B.dropna(axis=1, how='all')

# Drop rows after 6 hours of Sepsis to prevent data leakage
df_B['Sepsis_Duration'] = df_B.groupby('Patient_ID')['SepsisLabel'].cumsum()
df_B = df_B[df_B['Sepsis_Duration'] <= 6].copy()
df_B = df_B.drop(columns=['Sepsis_Duration'])
if 'ICULOS' in df_B.columns:
    df_B = df_B.drop(columns=['ICULOS'])


print("Engineering Missingness Flags...")
if 'Lactate' in df_B.columns:
    df_B['Lactate_Order_Count'] = df_B.groupby('Patient_ID')['Lactate'].transform(lambda x: x.notnull().cumsum())
if 'WBC' in df_B.columns:
    df_B['WBC_Order_Count'] = df_B.groupby('Patient_ID')['WBC'].transform(lambda x: x.notnull().cumsum())


print("--- 2. APPLYING HOSPITAL A's RULES ---")
# Load the Medians we saved from Hospital A (No data leakage!)
with open('../data/cleaned/hospital_A_medians.json', 'r') as f:
    medians_A = json.load(f)

# Only process columns that we actually have medians for
feature_cols = [col for col in medians_A.keys() if col in df_B.columns]

# Forward-fill patient-by-patient
print("Forward-filling vitals...")
df_B[feature_cols] = df_B.groupby('Patient_ID')[feature_cols].ffill()

# Fill any remaining NaNs with Hospital A's global medians
print("Applying Hospital A medians to Hospital B...")
for col in feature_cols:
    df_B[col] = df_B[col].fillna(medians_A[col])


print("--- 3. ENGINEERING PHYSIOLOGICAL FEATURES ---")
# 1. System Overload Score
df_B['System_Overload_Score'] = (
    (df_B['HR'] > 100).astype(int) +
    (df_B['SBP'] < 90).astype(int) +
    (df_B['Resp'] > 22).astype(int) +
    ((df_B['Temp'] > 38) | (df_B['Temp'] < 36)).astype(int)
)

# 2. Age-Adjusted Frailty
df_B['Age_Frailty_Index'] = (df_B['Age'] / 100) * df_B['System_Overload_Score']

# Vital Sign Dynamics
df_B['FiO2_4hr_mean'] = df_B.groupby('Patient_ID')['FiO2'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
df_B['FiO2_4hr_std'] = df_B.groupby('Patient_ID')['FiO2'].transform(lambda x: x.rolling(window=4, min_periods=2).std())
df_B['Temp_4hr_mean'] = df_B.groupby('Patient_ID')['Temp'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
df_B['SBP_4hr_mean'] = df_B.groupby('Patient_ID')['SBP'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
df_B['SBP_4hr_std'] = df_B.groupby('Patient_ID')['SBP'].transform(lambda x: x.rolling(window=4, min_periods=2).std())
df_B['SBP_Volatility'] = df_B['SBP_4hr_std'] / (df_B['SBP_4hr_mean'] + 1e-5)

# Domain-Agnostic Deltas
o2sat_baseline = df_B.groupby('Patient_ID')['O2Sat'].transform('first')
hr_baseline = df_B.groupby('Patient_ID')['HR'].transform('first')
resp_baseline = df_B.groupby('Patient_ID')['Resp'].transform('first')
map_baseline = df_B.groupby('Patient_ID')['MAP'].transform('first')

df_B['O2Sat_Admission_Delta'] = df_B['O2Sat'] - o2sat_baseline
df_B['HR_Admission_Delta'] = df_B['HR'] - hr_baseline
df_B['Resp_Admission_Delta'] = df_B['Resp'] - resp_baseline
df_B['MAP_Admission_Delta'] = df_B['MAP'] - map_baseline

# High-Frequency Rolling Vitals
df_B['HR_4hr_mean'] = df_B.groupby('Patient_ID')['HR'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
df_B['HR_4hr_std'] = df_B.groupby('Patient_ID')['HR'].transform(lambda x: x.rolling(window=4, min_periods=2).std())
df_B['MAP_4hr_mean'] = df_B.groupby('Patient_ID')['MAP'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())


bili_baseline = df_B.groupby('Patient_ID')['Bilirubin_total'].transform('first')
df_B['Bilirubin_Admission_Delta'] = df_B['Bilirubin_total'] - bili_baseline


print("--- 4. INITIATING DEPLOYMENT ---")
x_B = df_B.drop(columns=['SepsisLabel', 'Patient_ID'])
y_B = df_B['SepsisLabel']

print("Loading Final Tuned XGBoost Model...")
champion_model = xgb.XGBClassifier()
champion_model.load_model('../models/tuned_model.json') 

x_B = x_B[champion_model.feature_names_in_]

# Generate Predictions
y_pred_proba_B = champion_model.predict_proba(x_B)[:, 1]


LOCKED_THRESHOLD = 0.2475
print(f"Applying locked threshold: {LOCKED_THRESHOLD*100}%")
y_pred_binary = (y_pred_proba_B >= LOCKED_THRESHOLD).astype(int)

# Score the real-world performance
test_recall = recall_score(y_B, y_pred_binary)
test_precision = precision_score(y_B, y_pred_binary)
test_f2 = fbeta_score(y_B, y_pred_binary, beta=2)
tn, fp, fn, tp = confusion_matrix(y_B, y_pred_binary).ravel()

print("\n==============================================")
print("--- HOSPITAL B (UNSEEN DATA) FINAL RESULTS ---")
print("==============================================")
print(f"Deployment F2-Score:  {test_f2:.4f} (Validation Goal: ~0.1986)")
print(f"Deployment Precision: {test_precision:.4f} (Validation Goal: ~0.0602)")
print(f"Deployment Recall:    {test_recall:.4f} (Validation Goal: ~0.4665)")
print("----------------------------------------------")
print("--- REAL-WORLD IMPACT (CONFUSION MATRIX) ---")
print(f"True Positives (Caught Sepsis):     {tp}")
print(f"False Negatives (Missed Sepsis):    {fn}")
print(f"False Positives (False Alarms):     {fp}")
print(f"True Negatives (Safely Ignored):    {tn}")
print("==============================================")

--- 1. LOADING UNSEEN DATA (HOSPITAL B) ---
Engineering Missingness Flags...
--- 2. APPLYING HOSPITAL A's RULES ---
Forward-filling vitals...
Applying Hospital A medians to Hospital B...
--- 3. ENGINEERING PHYSIOLOGICAL FEATURES ---
--- 4. INITIATING DEPLOYMENT ---
Loading Final Tuned XGBoost Model...
Applying locked threshold: 24.75%

--- HOSPITAL B (UNSEEN DATA) FINAL RESULTS ---
Deployment F2-Score:  0.1518 (Validation Goal: ~0.1986)
Deployment Precision: 0.0471 (Validation Goal: ~0.0602)
Deployment Recall:    0.3418 (Validation Goal: ~0.4665)
----------------------------------------------
--- REAL-WORLD IMPACT (CONFUSION MATRIX) ---
True Positives (Caught Sepsis):     2328
False Negatives (Missed Sepsis):    4483
False Positives (False Alarms):     47111
True Negatives (Safely Ignored):    704104
